# EmbdGuard Validation on Amazon Reviews 2023 (Video Games)

Validates EmbdGuard's **6 detectors** and **3 defenses** at scale on Amazon Reviews 2023 Video Games (~95K users, ~26K items, ~815K interactions).

**Pipeline:**
1. Load Amazon Reviews, train clean baseline
2. Run DLAttack to generate poisoned training data
3. Retrain on poisoned data with **EmbdGuard monitoring** — all detectors active
4. Retrain on poisoned data with **EmbdGuard defenses** active — measure mitigation
5. Compare: no guard vs detection-only vs detection+defense

**Detectors:** GradientAnomaly, AccessFrequency, EmbeddingDrift, GradientDistribution, TemporalAccess, TIA

**Defenses:** GradientClip, RowFreeze, InteractionFilter

**Requirements:** GPU runtime (Runtime > Change runtime type > A100 GPU)

---

## Phase 0: Setup

In [ ]:
import os
if not os.path.exists("embdguard"):
    !git clone https://github.com/aliafzal/embdguard.git
%cd /content/embdguard
!git pull

In [ ]:
!pip install -q torchrec fbgemm-gpu datasets

In [ ]:
import sys, warnings
import torch
import numpy as np
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch:    {torch.__version__}")
print(f"device:   {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Import EmbdGuard modules (from repo root src/)
from src.models import build_ebc, TwoTower, TwoTowerTrainTask, make_kjt, make_optimizer
from src.guard import EmbdGuard
from src.detectors.gradient_anomaly import GradientAnomalyDetector
from src.detectors.access_frequency import AccessFrequencyDetector
from src.detectors.embedding_drift import EmbeddingDriftDetector
from src.detectors.gradient_distribution import GradientDistributionDetector
from src.detectors.temporal_access import TemporalAccessDetector
from src.defenses.gradient_clip import GradientClipDefense
from src.defenses.row_freeze import RowFreezeDefense
from src.defenses.interaction_filter import InteractionFilterDefense

# Save EmbdGuard src reference before importing dlattack modules
import src as embdguard_src
_saved_src_modules = {k: v for k, v in sys.modules.items() if k == "src" or k.startswith("src.")}

# Import dlattack modules (need path swap since they use `from src.train import ...`)
sys.path.insert(0, "dlattack_research")
for key in list(sys.modules.keys()):
    if key == "src" or (key.startswith("src.") and not key.startswith("src.models")):
        del sys.modules[key]

import src.dataset as dl_dataset
import src.train as dl_train
import src.attack as dl_attack
import src.evaluate as dl_evaluate

# Restore EmbdGuard src
for k, v in _saved_src_modules.items():
    sys.modules[k] = v
sys.path.remove("dlattack_research")

print("Imports OK")

In [ ]:
os.makedirs("checkpoints/amazon", exist_ok=True)
os.makedirs("results/amazon", exist_ok=True)

## Phase 1: Load Amazon Reviews 2023 (Video Games, 5-core)

In [ ]:
from datasets import load_dataset

def load_amazon_reviews(category="Video_Games", kcore=5, min_interactions=5):
    config = f"{kcore}core_rating_only_{category}"
    print(f"Loading Amazon Reviews 2023: {config} ...")
    ds = load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023", config,
        cache_dir="data/amazon", trust_remote_code=True,
    )
    split_name = list(ds.keys())[0]
    df = ds[split_name].to_pandas()
    print(f"  Raw: {len(df):,} interactions")

    df = df.rename(columns={"parent_asin": "item_id"})
    df = df[["user_id", "item_id", "rating", "timestamp"]].copy()

    counts = df.groupby("user_id")["item_id"].count()
    active = counts[counts >= min_interactions].index
    df = df[df["user_id"].isin(active)].copy()

    user_map = {u: i for i, u in enumerate(df["user_id"].unique())}
    item_map = {v: i for i, v in enumerate(df["item_id"].unique())}
    df["user_id"] = df["user_id"].map(user_map)
    df["item_id"] = df["item_id"].map(item_map)
    df["label"] = 1

    n_users = df["user_id"].nunique()
    n_items = df["item_id"].nunique()
    density = len(df) / (n_users * n_items) * 100
    print(f"  Users: {n_users:,}, Items: {n_items:,}, "
          f"Interactions: {len(df):,}, Density: {density:.4f}%")
    return df, n_users, n_items

df, n_users, n_items = load_amazon_reviews()
train_df, test_df = dl_dataset.split_data(df)
print(f"\nTrain: {len(train_df):,}  |  Test: {len(test_df):,}")

In [ ]:
interactions_per_user = train_df.groupby("user_id").size()
interactions_per_item = train_df.groupby("item_id").size()

print(f"Per user — mean: {interactions_per_user.mean():.1f}, "
      f"median: {interactions_per_user.median():.0f}, "
      f"min: {interactions_per_user.min()}, max: {interactions_per_user.max()}")
print(f"Per item — mean: {interactions_per_item.mean():.1f}, "
      f"median: {interactions_per_item.median():.0f}, "
      f"min: {interactions_per_item.min()}, max: {interactions_per_item.max()}")

## Phase 2: Train Clean Baseline

In [ ]:
# === CONFIGURATION ===
EMBEDDING_DIM = 64
LAYER_SIZES = [128, 64]
BASELINE_EPOCHS = 30
BATCH_SIZE = 8192
LR = 0.001
N_NEG = 4

In [ ]:
ebc = build_ebc(n_users, n_items, EMBEDDING_DIM, device=DEVICE)
two_tower = TwoTower(ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
model = TwoTowerTrainTask(two_tower)
optimizer = make_optimizer(model, lr=LR)

eval_fn = lambda m: dl_evaluate.evaluate(
    m, test_df, train_df, n_items, n_neg=99, k=10, device=str(DEVICE)
)

print(f"=== Baseline Training | {n_users:,} users, {n_items:,} items ===")
history = dl_train.train(
    model, optimizer, train_df, n_items,
    epochs=BASELINE_EPOCHS, batch_size=BATCH_SIZE,
    n_neg=N_NEG, device=str(DEVICE),
    eval_fn=eval_fn,
    save_path="checkpoints/amazon/baseline.pt",
)

In [ ]:
import matplotlib.pyplot as plt

epochs_list = [h[0] for h in history]
losses = [h[1] for h in history]
eval_epochs = [h[0] for h in history if h[2]]
hr_values = [h[2]["HR@K"] for h in history if h[2]]
ndcg_values = [h[2]["NDCG@K"] for h in history if h[2]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_list, losses, "b-o", markersize=3)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training Loss"); ax1.grid(True, alpha=0.3)

ax2.plot(eval_epochs, hr_values, "g-o", label="HR@10", markersize=5)
ax2.plot(eval_epochs, ndcg_values, "r-s", label="NDCG@10", markersize=5)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.set_title("Evaluation Metrics"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/amazon/baseline_training.png", dpi=150, bbox_inches="tight")
plt.show()

final = history[-1][2]
if final:
    print(f"\nBaseline: HR@10={final['HR@K']:.4f}, NDCG@10={final['NDCG@K']:.4f}")

## Phase 3: DLAttack — Generate Poisoned Data

Run DLAttack to create poisoned training data. This is the attack we'll then try to detect and defend against with EmbdGuard.

In [ ]:
# === ATTACK CONFIGURATION ===
ATTACK_ROUNDS = 5
FAKE_USERS_PER_ROUND = 10
N_FILLER = 50
N_OPTIM_STEPS = 300
RETRAIN_EPOCHS = 10

In [ ]:
# Select target item
item_counts = train_df["item_id"].value_counts()
mid_items = item_counts[(item_counts > 20) & (item_counts < 200)].index.tolist()
target_item = int(mid_items[0])
print(f"Target item: {target_item} ({item_counts[target_item]} interactions)")
print(f"Candidates: {len(mid_items)} mid-popularity items")

In [ ]:
import json

# Load baseline and run attack
attack_ebc = build_ebc(n_users, n_items, EMBEDDING_DIM, device=DEVICE)
attack_tt = TwoTower(attack_ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
attack_model = TwoTowerTrainTask(attack_tt)
state = torch.load("checkpoints/amazon/baseline.pt", map_location=DEVICE, weights_only=False)
attack_model.load_state_dict(state, strict=False)
attack_optimizer = make_optimizer(attack_model, lr=LR)

attack_results, poisoned_train, attack_model, attack_optimizer = dl_attack.run_dlattack(
    model=attack_model, optimizer=attack_optimizer,
    train_df=train_df, test_df=test_df,
    n_users=n_users, n_items=n_items,
    target_item_id=target_item, embedding_dim=EMBEDDING_DIM,
    layer_sizes=LAYER_SIZES, rounds=ATTACK_ROUNDS, m=FAKE_USERS_PER_ROUND,
    n_filler=N_FILLER, n_optim_steps=N_OPTIM_STEPS,
    retrain_epochs=RETRAIN_EPOCHS, lr=LR, device=str(DEVICE),
    eval_fn=eval_fn,
)

torch.save(attack_model.state_dict(), "checkpoints/amazon/attacked_model.pt")
poisoned_train.to_csv("results/amazon/poisoned_train.csv", index=False)
with open("results/amazon/attack_results.json", "w") as f:
    json.dump({"target_item": target_item, "n_users": n_users,
               "rounds": attack_results}, f, indent=2)

n_fake = ATTACK_ROUNDS * FAKE_USERS_PER_ROUND
print(f"\nAttack complete: {n_fake} fake users injected ({n_fake/n_users*100:.3f}% poison ratio)")

In [ ]:
# Baseline attack success (no EmbdGuard)
def _load_tt(ckpt):
    state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    tt_state = {k[len("two_tower."):]: v for k, v in state.items() if k.startswith("two_tower.")}
    if not tt_state:
        tt_state = state
    nu = tt_state["ebc.embedding_bags.t_user_id.weight"].shape[0]
    ni = tt_state["ebc.embedding_bags.t_item_id.weight"].shape[0]
    ebc = build_ebc(nu, ni, EMBEDDING_DIM, device=DEVICE)
    tt = TwoTower(ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
    tt.load_state_dict(tt_state, strict=False)
    return tt

clean_tt = _load_tt("checkpoints/amazon/baseline.pt")
attacked_tt = _load_tt("checkpoints/amazon/attacked_model.pt")

clean_metrics = dl_evaluate.evaluate(clean_tt, test_df, train_df, n_items, device=str(DEVICE))
attacked_metrics = dl_evaluate.evaluate(attacked_tt, test_df, train_df, n_items, device=str(DEVICE))
clean_thr = dl_evaluate.target_item_hit_ratio(clean_tt, target_item, test_df, train_df, n_items, device=str(DEVICE))
attacked_thr = dl_evaluate.target_item_hit_ratio(attacked_tt, target_item, test_df, train_df, n_items, device=str(DEVICE))

print("=" * 55)
print("  ATTACK SUCCESS (no EmbdGuard)")
print("=" * 55)
print(f"  {'':30s} {'Clean':>10} {'Attacked':>10}")
print(f"  {'HR@10':30s} {clean_metrics['HR@K']:>10.4f} {attacked_metrics['HR@K']:>10.4f}")
print(f"  {'NDCG@10':30s} {clean_metrics['NDCG@K']:>10.4f} {attacked_metrics['NDCG@K']:>10.4f}")
print(f"  {'Target Item HR@10':30s} {clean_thr:>10.4f} {attacked_thr:>10.4f}")
print("=" * 55)

## Phase 4: EmbdGuard Detection — All 5 Real-Time Detectors

Retrain from the baseline checkpoint on poisoned data with EmbdGuard monitoring attached. All 5 real-time detectors active:

| Detector | Signal |
|----------|--------|
| GradientAnomaly | Gradient norm z-score spike |
| AccessFrequency | Embedding row access concentration |
| EmbeddingDrift | Per-row weight drift from reference |
| GradientDistribution | Kurtosis + concentration of gradients |
| TemporalAccess | Jaccard overlap + burst repetition |

In [ ]:
# Build fresh model from baseline
det_ebc = build_ebc(n_users, n_items, EMBEDDING_DIM, device=DEVICE)
det_tt = TwoTower(det_ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
det_model = TwoTowerTrainTask(det_tt)

# Expand user embedding for fake users in poisoned data
max_uid = int(poisoned_train["user_id"].max()) + 1
expanded_ebc = build_ebc(max_uid, n_items, EMBEDDING_DIM, device=DEVICE)
expanded_tt = TwoTower(expanded_ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
dl_attack._copy_weights_to_plain(
    torch.load("checkpoints/amazon/baseline.pt", map_location=DEVICE, weights_only=False),
    expanded_tt,
)
det_model = TwoTowerTrainTask(expanded_tt)
det_optimizer = make_optimizer(det_model, lr=LR)

# Attach EmbdGuard with all 5 real-time detectors
guard = EmbdGuard(det_model, log_path="results/amazon/guard_detection.jsonl")
guard.add_detector(GradientAnomalyDetector(threshold_z=5.0, min_steps=50))
guard.add_detector(AccessFrequencyDetector(concentration_threshold=3.0, min_steps=30))
guard.add_detector(EmbeddingDriftDetector(
    drift_threshold_z=8.0, min_steps=50, snapshot_interval=100))
guard.add_detector(GradientDistributionDetector(
    kurtosis_z=5.0, concentration_threshold=50.0, min_steps=50))
guard.add_detector(TemporalAccessDetector(
    burst_window=10, burst_threshold=1.0, top_k=3, min_steps=50))

print(f"EmbdGuard attached with {len(guard._detectors)} detectors")
print(f"Training on {len(poisoned_train):,} interactions (poisoned)")

In [ ]:
# Retrain on poisoned data with EmbdGuard monitoring
pos_users = torch.tensor(poisoned_train["user_id"].values, dtype=torch.long, device=DEVICE)
pos_items = torch.tensor(poisoned_train["item_id"].values, dtype=torch.long, device=DEVICE)

all_alerts = []
alerts_by_detector = {}
loss_history = []
alert_timeline = []  # (step, detector_name)

step = 0
for epoch in range(1, RETRAIN_EPOCHS + 1):
    det_model.train()
    users, items, labels = dl_train._negative_sample_tensors(
        pos_users, pos_items, n_items, N_NEG, DEVICE
    )
    n_samples = len(users)
    batch_size = 4096
    total_loss = 0.0
    epoch_alerts = 0

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        kjt = make_kjt(users[start:end], items[start:end])
        batch_labels = labels[start:end]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            loss, _ = det_model(kjt, batch_labels)
            det_optimizer.zero_grad()
            loss.backward()
        det_optimizer.step()

        step += 1
        alerts = guard.step()
        all_alerts.extend(alerts)
        epoch_alerts += len(alerts)
        total_loss += loss.item() * (end - start)

        for a in alerts:
            alerts_by_detector.setdefault(a.detector, []).append(a)
            alert_timeline.append((step, a.detector))

    train_loss = total_loss / n_samples
    loss_history.append(train_loss)
    print(f"  Epoch {epoch:2d} | loss={train_loss:.4f} | alerts={epoch_alerts}")

guard.detach()
print(f"\nTotal alerts: {len(all_alerts)}")

In [ ]:
# Detection results summary
print("=" * 65)
print("  DETECTOR RESULTS (detection-only, no defense)")
print("=" * 65)
print(f"  {'Detector':<28} {'Alerts':>8} {'First Step':>12} {'Flagged Rows':>14}")
print(f"  {'-'*28} {'-'*8} {'-'*12} {'-'*14}")

detector_names = [
    "gradient_anomaly", "access_frequency", "embedding_drift",
    "gradient_distribution", "temporal_access",
]

flagged_items = set()  # items flagged across all detectors

for det_name in detector_names:
    det_alerts = alerts_by_detector.get(det_name, [])
    if det_alerts:
        first = min(a.step for a in det_alerts)
        rows = set()
        for a in det_alerts:
            row = a.details.get("row_id") or a.details.get("hottest_row")
            if row is not None:
                rows.add(int(row))
                flagged_items.add(int(row))
        row_str = str(sorted(rows)[:5]) if rows else "-"
        print(f"  {det_name:<28} {len(det_alerts):>8} {first:>12} {row_str:>14}")
    else:
        print(f"  {det_name:<28} {'0':>8} {'-':>12} {'-':>14}")

print(f"\n  Target item {target_item}: "
      f"{'DETECTED' if target_item in flagged_items else 'NOT detected'}")
print("=" * 65)

In [ ]:
# Alert timeline visualization
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Loss curve
ax1.plot(range(1, len(loss_history) + 1), loss_history, "b-o", markersize=4)
ax1.set_ylabel("Loss")
ax1.set_title("Poisoned Retraining Loss")
ax1.grid(True, alpha=0.3)

# Alert counts per epoch
epoch_steps = step // RETRAIN_EPOCHS
det_colors = {
    "gradient_anomaly": "red",
    "access_frequency": "orange",
    "embedding_drift": "purple",
    "gradient_distribution": "green",
    "temporal_access": "blue",
}
for det_name, color in det_colors.items():
    det_steps = [s for s, d in alert_timeline if d == det_name]
    if det_steps:
        ax2.scatter(det_steps, [det_name] * len(det_steps),
                    c=color, s=10, alpha=0.6, label=det_name)
ax2.set_xlabel("Training Step")
ax2.set_title("Alert Timeline by Detector")
ax2.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.savefig("results/amazon/detection_timeline.png", dpi=150, bbox_inches="tight")
plt.show()

## Phase 5: EmbdGuard Defense — Detection + Active Mitigation

Retrain again from the same baseline, but now with defenses active. When a detector fires, defenses automatically engage:

| Defense | Action |
|---------|--------|
| GradientClip | Clip gradients on flagged rows to max_norm |
| RowFreeze | Zero gradients on flagged rows (full freeze) |
| InteractionFilter | Remove flagged items from batches |

In [ ]:
# Build fresh model from baseline (same starting point)
def_ebc = build_ebc(max_uid, n_items, EMBEDDING_DIM, device=DEVICE)
def_tt = TwoTower(def_ebc, layer_sizes=LAYER_SIZES, device=DEVICE)
dl_attack._copy_weights_to_plain(
    torch.load("checkpoints/amazon/baseline.pt", map_location=DEVICE, weights_only=False),
    def_tt,
)
def_model = TwoTowerTrainTask(def_tt)
def_optimizer = make_optimizer(def_model, lr=LR)

# Attach EmbdGuard with detectors AND defenses
guard_def = EmbdGuard(def_model, log_path="results/amazon/guard_defense.jsonl")
guard_def.add_detector(GradientAnomalyDetector(threshold_z=5.0, min_steps=50))
guard_def.add_detector(AccessFrequencyDetector(concentration_threshold=3.0, min_steps=30))
guard_def.add_detector(EmbeddingDriftDetector(
    drift_threshold_z=8.0, min_steps=50, snapshot_interval=100))
guard_def.add_detector(GradientDistributionDetector(
    kurtosis_z=5.0, concentration_threshold=50.0, min_steps=50))
guard_def.add_detector(TemporalAccessDetector(
    burst_window=10, burst_threshold=1.0, top_k=3, min_steps=50))

# Add all 3 defenses
grad_clip = GradientClipDefense(max_norm=0.01)
row_freeze = RowFreezeDefense()
interaction_filter = InteractionFilterDefense()

guard_def.add_defense(grad_clip)
guard_def.add_defense(row_freeze)
guard_def.add_defense(interaction_filter)

print(f"EmbdGuard: {len(guard_def._detectors)} detectors + {len(guard_def._defenses)} defenses")

In [ ]:
# Retrain with defenses active
def_alerts = []
def_loss_history = []

step = 0
for epoch in range(1, RETRAIN_EPOCHS + 1):
    def_model.train()
    users, items, labels = dl_train._negative_sample_tensors(
        pos_users, pos_items, n_items, N_NEG, DEVICE
    )
    n_samples = len(users)
    batch_size = 4096
    total_loss = 0.0
    epoch_alerts = 0
    epoch_filtered = 0

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        b_users = users[start:end]
        b_items = items[start:end]
        b_labels = labels[start:end]

        # Apply interaction filter defense
        orig_len = len(b_users)
        b_users, b_items, b_labels = interaction_filter.filter_batch(
            b_users, b_items, b_labels
        )
        epoch_filtered += orig_len - len(b_users)

        if len(b_users) == 0:
            step += 1
            guard_def.step()
            continue

        kjt = make_kjt(b_users, b_items)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            loss, _ = def_model(kjt, b_labels)
            def_optimizer.zero_grad()
            loss.backward()
        def_optimizer.step()

        step += 1
        alerts = guard_def.step()
        def_alerts.extend(alerts)
        epoch_alerts += len(alerts)
        total_loss += loss.item() * len(b_users)

    train_loss = total_loss / max(n_samples - epoch_filtered, 1)
    def_loss_history.append(train_loss)
    print(f"  Epoch {epoch:2d} | loss={train_loss:.4f} | "
          f"alerts={epoch_alerts} | filtered={epoch_filtered}")

guard_def.detach()
print(f"\nTotal alerts: {len(def_alerts)}")

## Phase 6: TIA Detector (Post-hoc)

TIA runs post-hoc on the poisoned data to identify fake users by their interaction profiles.

In [ ]:
from src.detectors.tia import TIADetector

# Run TIA on the detection-only model (retrained on poisoned data)
det_inner = det_model.two_tower if hasattr(det_model, "two_tower") else det_model

tia = TIADetector(
    watch_items=[target_item],
    train_df=poisoned_train,
    top_similar=50,
    threshold_percentile=95.0,
)

# TIA check needs table_stats and model — use guard's stats
# But since guard is detached, run TIA directly
fake_user_ids = set(poisoned_train[poisoned_train["user_id"] >= n_users]["user_id"].unique())

# Use the underlying anomaly scoring from dlattack's detect module
sys.path.insert(0, "dlattack_research")
del sys.modules["src"]
for key in list(sys.modules.keys()):
    if key.startswith("src."):
        del sys.modules[key]
import src.detect as dl_detect
sys.modules["src"] = embdguard_src
sys.path.remove("dlattack_research")

scores = dl_detect.compute_user_anomaly_scores(poisoned_train, det_inner, target_item, n_items)
real_scores = scores[~scores.index.isin(fake_user_ids)]
fake_scores = scores[scores.index.isin(fake_user_ids)]

print(f"TIA Anomaly Scores:")
print(f"  Real users — mean: {real_scores.mean():.4f}, std: {real_scores.std():.4f}")
print(f"  Fake users — mean: {fake_scores.mean():.4f}, std: {fake_scores.std():.4f}")
print(f"  Separation: {fake_scores.mean() / max(real_scores.mean(), 1e-8):.1f}x")

# Precision/recall at thresholds
print(f"\n  {'Threshold':<10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print(f"  {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
for p in [90, 95, 99]:
    threshold = np.percentile(scores.values, p)
    flagged = set(scores[scores >= threshold].index.tolist())
    tp = len(flagged & fake_user_ids)
    precision = tp / max(len(flagged), 1)
    recall = tp / max(len(fake_user_ids), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    print(f"  p{p:<8} {precision:>10.4f} {recall:>10.4f} {f1:>10.4f}")

In [ ]:
# TIA score distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(real_scores, bins=50, alpha=0.7,
        label=f"Real users (n={len(real_scores):,})", color="steelblue", density=True)
ax.hist(fake_scores, bins=20, alpha=0.7,
        label=f"Fake users (n={len(fake_scores)})", color="indianred", density=True)
ax.set_xlabel("TIA Anomaly Score")
ax.set_ylabel("Density")
ax.set_title("TIA Score Distribution — Amazon Video Games")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/amazon/tia_scores.png", dpi=150, bbox_inches="tight")
plt.show()

## Phase 7: Final Comparison — No Guard vs Detection vs Defense

In [ ]:
# Evaluate all three models
det_inner = det_model.two_tower if hasattr(det_model, "two_tower") else det_model
def_inner = def_model.two_tower if hasattr(def_model, "two_tower") else def_model

configs = [
    ("Clean Baseline", clean_tt),
    ("Attacked (no guard)", attacked_tt),
    ("Attacked + Detection", det_inner),
    ("Attacked + Defense", def_inner),
]

print("=" * 70)
print("  FINAL COMPARISON — EmbdGuard on Amazon Video Games")
print("=" * 70)
print(f"  {'Config':<28} {'HR@10':>10} {'NDCG@10':>10} {'Tgt HR@10':>10}")
print(f"  {'-'*28} {'-'*10} {'-'*10} {'-'*10}")

results_table = []
for label, tt in configs:
    m = dl_evaluate.evaluate(tt, test_df, train_df, n_items, device=str(DEVICE))
    thr = dl_evaluate.target_item_hit_ratio(
        tt, target_item, test_df, train_df, n_items, device=str(DEVICE)
    )
    print(f"  {label:<28} {m['HR@K']:>10.4f} {m['NDCG@K']:>10.4f} {thr:>10.4f}")
    results_table.append({"config": label, "HR@10": m["HR@K"],
                          "NDCG@10": m["NDCG@K"], "Target_HR@10": thr})

print("=" * 70)

# Defense effectiveness
no_guard_thr = results_table[1]["Target_HR@10"]
defended_thr = results_table[3]["Target_HR@10"]
if no_guard_thr > 0:
    reduction = (1 - defended_thr / no_guard_thr) * 100
    print(f"\n  Defense reduced target promotion by {reduction:.1f}%")
print(f"  Total real-time alerts fired: {len(all_alerts)} (detect) / {len(def_alerts)} (defend)")

In [ ]:
# Comparison bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

labels = [r["config"] for r in results_table]
hr_vals = [r["HR@10"] for r in results_table]
thr_vals = [r["Target_HR@10"] for r in results_table]
colors = ["steelblue", "indianred", "orange", "seagreen"]

ax1.bar(range(len(labels)), hr_vals, color=colors)
ax1.set_xticks(range(len(labels)))
ax1.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax1.set_ylabel("HR@10")
ax1.set_title("Overall Recommendation Quality")
ax1.grid(True, alpha=0.3, axis="y")

ax2.bar(range(len(labels)), thr_vals, color=colors)
ax2.set_xticks(range(len(labels)))
ax2.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax2.set_ylabel("Target Item HR@10")
ax2.set_title(f"Target Item Promotion (item {target_item})")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("results/amazon/final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

In [ ]:
summary = {
    "dataset": "Amazon Reviews 2023 — Video_Games (5-core)",
    "n_users": n_users,
    "n_items": n_items,
    "target_item": target_item,
    "attack": {
        "rounds": ATTACK_ROUNDS,
        "fake_users": ATTACK_ROUNDS * FAKE_USERS_PER_ROUND,
        "poison_ratio_pct": ATTACK_ROUNDS * FAKE_USERS_PER_ROUND / n_users * 100,
    },
    "results": results_table,
    "detection": {
        "total_alerts": len(all_alerts),
        "by_detector": {
            det: len(alerts) for det, alerts in alerts_by_detector.items()
        },
        "target_item_flagged": target_item in flagged_items,
    },
    "defense": {
        "total_alerts": len(def_alerts),
    },
}

with open("results/amazon/embdguard_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

In [ ]:
try:
    from google.colab import files
    import shutil
    shutil.make_archive("amazon_embdguard_results", "zip", "results/amazon")
    files.download("amazon_embdguard_results.zip")
except ImportError:
    print("Not on Colab — results saved in results/amazon/")